# 02 — Native 5-minute volatility dynamics

## Independence contract

This module starts from the native five-minute return series and does **not** consume Module 01 outputs or conclusions. It also consumes nothing from any other research branch.

The only database columns read are:

- `state_time`
- `underlying`
- `return_5m`

The primitive volatility object is exactly:

\[
v_t=r_t^2,
\]

where (r_t) is the single native five-minute return.

There is **no rolling 30m/60m block**, no HAR model, no clock adjustment inherited from elsewhere, no IV, no order book, no flow variable, and no latent-state label.

## Questions

1. What is the distribution of single-bar five-minute variance?
2. Is five-minute volatility persistent from one native bar to later native bars?
3. Does a large five-minute variance bar make another large bar more likely next?
4. Do extreme five-minute bars cluster into consecutive runs beyond the one-bar timescale?
5. How often do extreme five-minute bars occur simultaneously across assets?
6. Does the native five-minute series itself have deterministic UTC clock structure?


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 260)
pd.set_option('display.max_rows', 400)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_dynamics_5m as vd5

START = os.getenv('VD5_START', '').strip() or None
END = os.getenv('VD5_END', '').strip() or None
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD5_ASSETS', '').split(',') if x.strip()]
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '02'
OUT.mkdir(parents=True, exist_ok=True)

print('volatility_dynamics_5m', vd5.VOLATILITY_DYNAMICS_5M_VERSION)
print('selftest:', vd5.selftest())
print('window:', START or 'earliest available', '->', END or 'latest available')


## 1. Load the native five-minute return series

No derived research feature is read from the database.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
available = set(con.execute('DESCRIBE time_series_feature_state').df()['column_name'])
required = {'state_time','underlying','return_5m'}
missing = sorted(required - available)
if missing:
    raise RuntimeError(f'time_series_feature_state lacks required columns: {missing}')

where = []
if START:
    where.append(f"state_time >= TIMESTAMPTZ '{pd.Timestamp(START, tz='UTC').isoformat()}'")
if END:
    end_ts = pd.Timestamp(END, tz='UTC') + pd.Timedelta(days=1)
    where.append(f"state_time < TIMESTAMPTZ '{end_ts.isoformat()}'")
if ASSET_FILTER:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
df = con.execute(
    'SELECT state_time, underlying, return_5m FROM time_series_feature_state'
    + where_sql + ' ORDER BY underlying, state_time'
).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
if df.empty:
    raise RuntimeError('No rows found.')
dupes = int(df.duplicated(['underlying','state_time']).sum())
if dupes:
    raise RuntimeError(f'Found {dupes} duplicate asset/timestamp rows.')
print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())}')
print(df.state_time.min(), '->', df.state_time.max())


## 2. Build the five-minute volatility panel

For each row:

\[
v_t=r_t^2.
\]

`log_variance_5m` is only a numerical transform for scale compression; its epsilon is estimated independently per asset from that asset's own positive five-minute variance observations.


In [ ]:
panels, metas = [], []
for asset, g in df.groupby('underlying', sort=True):
    p, meta = vd5.build_5m_panel(g)
    panels.append(p)
    metas.append(meta.__dict__)
panel = pd.concat(panels, ignore_index=True)
panel_meta = pd.DataFrame(metas)
display(panel_meta)


## 3. Data coverage and missingness

Before studying persistence or duration, check whether missing returns are concentrated at particular UTC times or dates. A missing native bar breaks any consecutive-bar sequence.


In [ ]:
missing_clock = vd5.missingness_by_clock(panel)
missing_date = vd5.missingness_by_date(panel)
display(panel_meta[['underlying','n_rows','n_finite_returns','finite_return_share']])
display(missing_date.sort_values(['underlying','coverage']).head(50))


## 4. Distribution of native five-minute variance

No smoothing or temporal aggregation is used.


In [ ]:
dist = vd5.distribution_summary(panel)
display(dist.round(12))

for asset in sorted(panel.underlying.unique()):
    x = panel.loc[panel.underlying == asset, 'log_variance_5m'].dropna()
    if len(x):
        fig, ax = plt.subplots(figsize=(8,3))
        ax.hist(x, bins=70)
        ax.set_title(f'{asset}: native 5m log variance')
        ax.set_xlabel('log(return_5m² + epsilon)')
        plt.show()


## 5. Native five-minute clock structure

This is measured here rather than assumed. Each exact UTC five-minute slot is compared only through the single-bar variance observed at that slot.


In [ ]:
clock_profile, clock_summary = vd5.clock_profile(panel)
display(clock_summary.round(4))

for asset in sorted(panel.underlying.unique()):
    g = clock_profile[clock_profile.underlying == asset].sort_values('clock_slot')
    if len(g):
        fig, ax = plt.subplots(figsize=(10,3))
        ax.plot(g.clock_slot, g['median'])
        ax.set_title(f'{asset}: native 5m volatility by UTC clock slot')
        ax.set_xlabel('5-minute UTC clock slot')
        ax.set_ylabel('median log 5m variance')
        plt.show()


## 6. Five-minute volatility persistence

We correlate one native bar's log variance with later **native bars** at exact timestamp lags. A 30-minute lag here means six five-minute bars apart; it does **not** mean a 30-minute realized-variance block.


In [ ]:
persistence = vd5.lag_persistence(panel)
display(persistence.pivot(index='underlying', columns='lag_minutes', values='spearman_log_variance').round(3))
display(persistence.pivot(index='underlying', columns='lag_minutes', values='pearson_log_variance').round(3))


## 7. One-step volatility-quantile transitions

Each asset's observed five-minute variance is ranked into ten empirical bins. We estimate the transition matrix from bar (t) to the immediately following five-minute bar (t+1), requiring an exact five-minute timestamp step.


In [ ]:
transitions = vd5.quantile_transition_matrix(panel, n_bins=10)
for asset in sorted(panel.underlying.unique()):
    g = transitions[transitions.underlying == asset]
    mat = g.pivot(index='from_bin', columns='to_bin', values='probability')
    print('\n', asset)
    display(mat.round(3))


## 8. Extreme native five-minute bars

We avoid committing to one threshold. For each asset, separately analyze its empirical 90th, 95th and 99th percentile of **single-bar** five-minute variance.

Two questions matter:

1. (P(v_{t+1}\text{ remains extreme}\mid v_t\text{ extreme})) versus the unconditional next-bar extreme probability;
2. how many consecutive native five-minute bars remain above the threshold.


In [ ]:
thresholds = vd5.tail_thresholds(panel)
tail_next = vd5.next_bar_tail_summary(panel, thresholds)
episodes = vd5.extract_tail_episodes(panel, thresholds)
episode_summary = vd5.episode_summary(episodes)
display(tail_next.round(4))
display(episode_summary.round(4))


## 9. Native five-minute paths around extreme-run onset

These paths contain only the single-bar (r_t^2) series. There is no rolling statistic to mechanically keep volatility elevated after a shock.


In [ ]:
paths = vd5.event_path_summary(panel, episodes, pre_bars=12, post_bars=12)
for q in sorted(paths['quantile'].unique()) if len(paths) else []:
    print('\nquantile', q)
    for asset in sorted(panel.underlying.unique()):
        g = paths[(paths.underlying == asset) & (paths.quantile == q)].sort_values('rel_bars')
        if not len(g):
            continue
        fig, ax = plt.subplots(figsize=(8,3))
        ax.plot(g.rel_min, g['median'], marker='o', markersize=2)
        ax.fill_between(g.rel_min, g.q25, g.q75, alpha=0.2)
        ax.axvline(0, linewidth=1)
        ax.set_title(f'{asset}: native 5m log variance around q={q:.2f} run onset')
        ax.set_xlabel('minutes relative to onset')
        ax.set_ylabel('log five-minute variance')
        plt.show()


## 10. Cross-asset synchronization

At each exact five-minute timestamp, count how many assets simultaneously exceed their own 90th/95th/99th-percentile single-bar variance thresholds.


In [ ]:
sync = vd5.synchronous_tail_events(panel, thresholds)
if len(sync):
    sync_summary = (
        sync.groupby(['quantile','n_assets']).size()
        .rename('n_timestamps').reset_index()
    )
else:
    sync_summary = pd.DataFrame(columns=['quantile','n_assets','n_timestamps'])
display(sync_summary)
display(sync[sync.n_assets >= 3].sort_values(['quantile','n_assets','state_time'], ascending=[True,False,True]).head(100) if len(sync) else sync)


## 11. Persist compact branch-local outputs


In [ ]:
panel_meta.to_csv(OUT / '02_panel_metadata.csv', index=False)
missing_clock.to_csv(OUT / '02_missingness_by_clock.csv', index=False)
missing_date.to_csv(OUT / '02_missingness_by_date.csv', index=False)
dist.to_csv(OUT / '02_5m_distribution.csv', index=False)
clock_profile.to_csv(OUT / '02_5m_clock_profile.csv', index=False)
clock_summary.to_csv(OUT / '02_5m_clock_effect_summary.csv', index=False)
persistence.to_csv(OUT / '02_5m_persistence.csv', index=False)
transitions.to_csv(OUT / '02_5m_quantile_transitions.csv', index=False)
thresholds.to_csv(OUT / '02_5m_tail_thresholds.csv', index=False)
tail_next.to_csv(OUT / '02_5m_tail_nextbar.csv', index=False)
episodes.to_csv(OUT / '02_5m_tail_episodes.csv', index=False)
episode_summary.to_csv(OUT / '02_5m_episode_summary.csv', index=False)
paths.to_csv(OUT / '02_5m_event_paths.csv', index=False)
sync.to_csv(OUT / '02_5m_synchronous_events.csv', index=False)
sync_summary.to_csv(OUT / '02_5m_synchronous_summary.csv', index=False)

meta = {
    'version': vd5.VOLATILITY_DYNAMICS_5M_VERSION,
    'module': '02_native_5m_volatility',
    'independence_contract': 'No outputs or assumptions from Module 01 or any other research branch.',
    'database_relation': 'time_series_feature_state',
    'database_columns_read': ['state_time','underlying','return_5m'],
    'primitive_volatility_object': 'variance_5m = return_5m ** 2',
    'rolling_aggregation_used': False,
    'predictive_model_used': False,
    'start': START or 'earliest_available',
    'end': END or 'latest_available',
    'assets': sorted(panel.underlying.unique().tolist()),
    'n_rows': int(len(panel)),
    'tail_quantiles': [0.90,0.95,0.99],
}
(OUT / '02_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote native 5m Module 02 outputs to', OUT)


## Interpretation boundary

This module can establish whether **native five-minute volatility** is heavy-tailed, clock-dependent, serially dependent, tail-persistent, consecutively clustered, and synchronized across assets.

It does not establish a forecast model, a latent regime, an HSMM duration law, MOM/MR/Bridge behavior, or an option-pricing implication.
